# 03 - XGBoost 基线模型

**目标**: 用 XGBoost 训练一个快速基线, 建立性能参考。

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style("whitegrid")
%matplotlib inline

## 1. 加载预处理好的数据

In [ ]:
from src.data.preprocess import CLASS_NAMES
from src.models.baseline import train_xgboost_baseline, evaluate

# 加载数据
train = np.load("../data/processed/train.npz")
val = np.load("../data/processed/val.npz")
test = np.load("../data/processed/test.npz")

X_train, y_train = train["X"], train["y"]
X_val, y_val = val["X"], val["y"]
X_test, y_test = test["X"], test["y"]

# 加载 label_encoder
with open("../data/processed/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

class_names = [CLASS_NAMES.get(c, f"class_{c}") for c in label_encoder.classes_]
num_classes = len(class_names)

print(f"类别数: {num_classes}")
print(f"训练: {X_train.shape}, 验证: {X_val.shape}, 测试: {X_test.shape}")

## 2. 训练 XGBoost

In [ ]:
model = train_xgboost_baseline(
    X_train, y_train,
    X_val, y_val,
    num_classes=num_classes
)

# 评估
train_metrics = evaluate(model, X_train, y_train)
val_metrics = evaluate(model, X_val, y_val)
test_metrics = evaluate(model, X_test, y_test)

print(f"{'':>10}  Accuracy  Macro-F1  Weighted-F1")
print(f"{'Train':>10}  {train_metrics['accuracy']:.4f}    {train_metrics['macro_f1']:.4f}    {train_metrics['weighted_f1']:.4f}")
print(f"{'Val':>10}  {val_metrics['accuracy']:.4f}    {val_metrics['macro_f1']:.4f}    {val_metrics['weighted_f1']:.4f}")
print(f"{'Test':>10}  {test_metrics['accuracy']:.4f}    {test_metrics['macro_f1']:.4f}    {test_metrics['weighted_f1']:.4f}")

## 3. 各类别性能

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print("分类报告:")
print(classification_report(
    y_test, y_pred,
    target_names=[f"{c}:{CLASS_NAMES.get(c, '?')[:15]}" for c in label_encoder.classes_],
    digits=3
))

## 4. 混淆矩阵

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize="true")
short_names = [CLASS_NAMES.get(c, str(c))[:20] for c in label_encoder.classes_]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=short_names, yticklabels=short_names,
            ax=ax, cbar_kws={"shrink": 0.8})
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("XGBoost 混淆矩阵 (归一化)", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 5. 特征重要性

In [ ]:
# XGBoost 特征重要性
importance = model.feature_importances_
feat_names = [f"feat_{i}" for i in range(len(importance))]

# 加载原始特征名
if os.path.exists("../data/processed/train.npz"):
    # 尝试从 scaler 获取特征名
    pass

sorted_idx = np.argsort(importance)[::-1]

fig, ax = plt.subplots(figsize=(8, 8))
top_n = 15
ax.barh(range(top_n), importance[sorted_idx[:top_n]][::-1], color="steelblue")
ax.set_yticks(range(top_n))
ax.set_yticklabels([feat_names[i] for i in sorted_idx[:top_n]][::-1])
ax.set_xlabel("Importance")
ax.set_title("XGBoost Top-15 特征重要性")
plt.tight_layout()
plt.show()

## 6. 错误分析

In [ ]:
# 找出模型最常出错的类别对
errors = y_pred != y_test
print(f"测试集错误数: {errors.sum()} / {len(y_test)} ({errors.mean():.2%})")

# 置信度最低的正确预测 vs 错误预测
max_probs = np.max(y_prob, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(max_probs[~errors], bins=30, alpha=0.7, label="正确", color="green")
axes[0].hist(max_probs[errors], bins=30, alpha=0.7, label="错误", color="red")
axes[0].set_xlabel("Max Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("预测置信度分布")
axes[0].legend()

# 各类别错误率
error_rates = []
for cls in sorted(set(y_test)):
    mask = y_test == cls
    error_rates.append((mask & errors).sum() / mask.sum())

axes[1].bar(range(len(error_rates)), error_rates, color="coral")
axes[1].set_xticks(range(len(error_rates)))
axes[1].set_xticklabels(short_names, rotation=45, ha="right", fontsize=7)
axes[1].set_ylabel("Error Rate")
axes[1].set_title("各类别错误率")

plt.tight_layout()
plt.show()

## 7. 保存模型

In [ ]:
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/xgboost_baseline.pkl")
print("模型已保存到 ../models/xgboost_baseline.pkl")

## 下一步

- **04_lstm_model**: 用 LSTM 直接从光变曲线序列学习, 对比 XGBoost 性能